### Vector Store


vector stores are specialized data stores that enable indexing and retrieving information based on vector representations.

These vectors called embeddings capture the semantic meaning of data that has been embedded.Vector stores are frequently used to search over unstructured data such as text , images ,audio to retrieve relevant information based on semantic similarity rather than exact keyword matches.

Langchain has a large number of vectorstore integration allowing users to easily switch between different vectorstore implementations.

### Key methods are :
1) add_documents : Add a list of texts to the vector store.
2) delete  :Delete a list of documents from the vector store.
3) similarity_search : search for similar documents to a given query.

LangChain supports integrations with dozens of vector stores, which can be broken down into three main categories based on how they are deployed and managed.
## 1. Local & In-Memory Vector Stores
Ideal for lightweight applications, quick prototyping, or running entirely on a local machine.
### FAISS: 
Developed by Meta, it runs entirely in-memory and is incredibly fast for semantic searches with smaller datasets.
### Chroma: 
A lightweight, open-source embedded vector database that is very simple to set up and works well for quick local development.
### Annoy: 
A C++ library with Python bindings that excels at memory efficiency and fast lookups.

## 2. Open-Source Dedicated Vector Databases
Designed for production environments, scalability, and handling larger datasets while managing persistence and filtering.
### Milvus: 
An open-source database built for massive, billion-scale datasets and high-throughput workloads.
### Weaviate: 
An open-source, GraphQL-native vector store that is highly customizable and handles complex metadata filtering easily.
### Qdrant: 
Known for its rich metadata filtering and performance speed.
## 3. Managed & Cloud-Native Services
Fully managed, highly scalable cloud solutions designed so you don't have to worry about infrastructure, scaling, or maintenance.
### Pinecone: 
A fully managed, cloud-native vector database widely used in production enterprise applications for its easy scalability.
### Elasticsearch: 
A traditional enterprise search engine that includes native vector search capabilities, allowing you to combine keyword search with semantic search.
### MongoDB Atlas Vector Search: 
Uses the popular MongoDB database to store and index vectors alongside your standard document data.
### Amazon OpenSearch: 
An AWS-native service well-suited for organizations already operating within the Amazon cloud ecosystem.
## 4. Relational & Standard Database Extensions
Ideal if you already use these databases and want to add vector search without introducing a completely new database system into your stack.
### pgvector: 
An open-source extension for PostgreSQL that allows you to store and query high-dimensional vector embeddings right inside a standard relational database.LangChain provides a standardized API for all these stores, meaning you can swap out your backend database with minimal code changes if your scalability needs grow.

### Choma DB

#### Adding Text

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv


load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

text_docs = ["Hello World", "World Economy"]

# Create a Chroma vector store from the text documents and embeddings and specify the collection name and persist directory for the database
ch_text_db = Chroma.from_texts(texts=text_docs, embedding=embeddings, collection_name="My_text", persist_directory="Chroma_db")

ch_text_db.persist() # Persist the vector store to disk kind of commit the changes to the database

C:\Users\Shrinath\AppData\Local\Temp\ipykernel_10348\1262128484.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
d:\LangChainP\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Shrinath\AppData\Local\Temp\ipykernel_10348\1262128484.py:15: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  ch_text_db.persist() # Persist the vector store to disk kind of commit the changes to the database


In [3]:
print("Vector store created and persisted successfully!")
print(ch_text_db.get()['documents'])

Vector store created and persisted successfully!
['Hello World', 'World Economy']


In [4]:
### if you want to add some additional documents to the existing vector store you can do it like this
text_docs2 = ["Data Science", "Machine Learning"]
ch_text_db.add_texts(texts=text_docs2)

print(ch_text_db.get()['documents'])

['Hello World', 'World Economy', 'Data Science', 'Machine Learning']


## Adding Documents :

In [ ]:
from importlib import metadata
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document # for creating documents with metadata
from dotenv import load_dotenv


load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
# Create a list of Document objects with page content and metadata
docs = [
  Document(page_content="Hello World", metadata={"source": "doc1"}),
  Document(page_content="World Economy", metadata={"source": "do2"})
] 

ch_doc_db = Chroma.from_documents(documents=docs, embedding=embeddings, collection_name="My_docs", persist_directory="Chroma_db")
ch_doc_db.persist()
print(ch_doc_db.get()["documents"])

['Hello World', 'World Economy']


In [6]:
### If you want to add more documents with metadata to the existing vector store you can do it like this
docs2 = [
  Document(page_content="Data Science", metadata={"source": "doc3"}),
  Document(page_content="Machine Learning", metadata={"source": "doc4"})
]

ch_doc_db.add_documents(documents=docs2)
print(ch_doc_db.get()["documents"])

['Hello World', 'World Economy', 'Data Science', 'Machine Learning']


## Similarity Search

In [8]:
# Now you can perform similarity search on the vector store and it will return the most similar documents along with their metadata
# here we are searching for the term "World" and asking for the top 2 most similar documents
result = ch_doc_db.similarity_search("World", k=2)
for i in result:
  print(i.page_content, i.metadata)

World Economy {'source': 'do2'}
Hello World {'source': 'doc1'}


# FAISS

## Adding text

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv


load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

text_docs = ["Hello World 2", "World Economy 2"]

# Create a faiss vector store from the text documents and embeddings.
# Faiss does not have an inbuilt persistence mechanism like chroma so you will have to save the index and the corresponding metadata separately using pickle or any other method of your choice
# it gets save in memory (eg. RAM) and you can use it for similarity search.
faiss_text_db = FAISS.from_texts(texts=text_docs, embedding=embeddings)


In [10]:
# if you want to see the documents in the vector store you can do it like this
# this will print the page content of all the documents in the vector store. 
# Note that since FAISS does not store the metadata so you will only get the page content and not the metadata of the documents in the vector store.
result1 = list(faiss_text_db.docstore._dict.values())
for d in result1:
  print(d.page_content)

Hello World 2
World Economy 2


In [12]:
# If you want to add more documents to the existing FAISS vector store you can do it like this
text_docs2 = ["Data Science 2 ", "Machine Learning 2"]

# in faiss there is no function to add more documents to the existing vector store like chroma 
# so you will have to create a new vector store with the new documents.
# so basically you will have to create a new vector store with the new documents and then merge the two vector stores together to create a single vector store with all the documents.
faiss_text_db2 = FAISS.from_texts(texts=text_docs2, embedding=embeddings)

In [13]:
result2 = list(faiss_text_db2.docstore._dict.values())
for d in result2:
  print(d.page_content)

Data Science 2 
Machine Learning 2


In [14]:
# you can merge the two vector stores together to create a single vector store with all the documents using the merge_from function
faiss_text_db2.merge_from(faiss_text_db)

In [15]:
result3 = list(faiss_text_db2.docstore._dict.values())
for d in result3:
  print(d.page_content)

Data Science 2 
Machine Learning 2
Hello World 2
World Economy 2


## Adding Document

In [16]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document # for creating documents with metadata
from dotenv import load_dotenv

load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

docs = [
  Document(page_content="Hello World 2", metadata={"source": "doc1"}),
  Document(page_content="World Economy 2", metadata={"source": "doc2"})
  
]

faiss_doc_db = FAISS.from_documents(documents=docs, embedding=embeddings)

In [17]:
result4 = list(faiss_doc_db.docstore._dict.values())
for d in result4:
  print(d.page_content)

Hello World 2
World Economy 2


In [18]:
# if you want to add more documents to the existing faiss vector store you can do it like this
docs2 = [
  Document(page_content="Data Science 2", metadata={"source": "doc3"}),
  Document(page_content="Machine Learning 2", metadata={"source": "doc4"})  
]

faiss_doc_db2 = FAISS.from_documents(documents=docs2, embedding=embeddings)

faiss_doc_db2.merge_from(faiss_doc_db)

result5 = list(faiss_doc_db2.docstore._dict.values())
for d in result5:
  print(d.page_content)

Data Science 2
Machine Learning 2
Hello World 2
World Economy 2


In [19]:
# perform the similarity search on the faiss vector store and it will return the most similar documents but without the metadata since faiss does not store the metadata of the documents in the vector store.
result6 =faiss_doc_db2.similarity_search("Science", k=2)
for i in result6:
  print(i.page_content, i.metadata)

Data Science 2 {'source': 'doc3'}
Machine Learning 2 {'source': 'doc4'}
